In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab.models.galaxy_zoo import (
    GalaxyZooData,
    GalaxyZooImages,
    GalaxyZooDataset,
)
from ugdatalab.models.galaxy_zoo.constants import N_LABELS, LABEL_COLUMNS, LABEL_DESCRIPTIVE
from ugdatalab.models.galaxy_zoo.images import _load_image
from ugdatalab.methods.neural_network.cnn import baseline_rmse

import plotters

# Galaxy Image Classification — Preprocessing

This notebook handles image preprocessing (Tasks 10–13):
1. **Task 10** — Crop and resize images to reduce memory by ~30×
2. **Task 11** — Set up efficient batch loading via PyTorch DataLoader
3. **Task 12** — Split into 80% training / 20% validation and verify label distributions
4. **Task 13** — Establish baseline model (mean prediction) RMSE

In [ ]:
CSV_PATH = Path("data/training_classifications.csv")
IMAGE_DIR = Path("data/training_images")

gz = GalaxyZooData(CSV_PATH)
labels_data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = labels_data["labels"]
galaxy_ids = labels_data["galaxy_ids"]

print(f"Galaxies: {len(gz)}")
print(f"Labels shape: {labels.shape}")

## Task 10 — Image Downsizing

The raw SDSS images are ~424×424 pixels, most of which is empty sky. We reduce memory and computation by:

1. **Center-cropping**: removing 25% of the border on each side (keeping the central 50%, which contains the galaxy). This is justified because SDSS cutouts are centered on the target and the outer border is almost always empty sky.
2. **Resampling**: resizing the cropped image to a smaller pixel grid.

The model input size is **96×96**. We cache images at the slightly larger **136×136** ($96 \times \sqrt{2}$, rounded up): this is the smallest size for which an arbitrary rotation of the cached image still fully covers a centered 96×96 inscribed square. Training and evaluation pipelines apply a downstream `CenterCrop(96)` (composed after any rotation augmentation), so the model always sees corner-artifact-free 96×96 input. We use 96 for the model rather than the nominal 64 so that ResNet-18 keeps its standard 7×7 stem; 96 also divides cleanly through four pooling stages (96 → 48 → 24 → 12 → 6).

In [ ]:
C
R
O
P
_
F
R
A
C
T
I
O
N
 
=
 
0
.
2
5

#
 
R
o
t
a
t
i
o
n
-
s
a
f
e
 
c
a
c
h
e
 
s
i
z
e
:
 
c
e
i
l
(
9
6
 
*
 
s
q
r
t
(
2
)
)
.
 
D
o
w
n
s
t
r
e
a
m
 
t
r
a
n
s
f
o
r
m
s

#
 
r
o
t
a
t
e
 
t
h
e
n
 
C
e
n
t
e
r
C
r
o
p
(
9
6
)
,
 
g
u
a
r
a
n
t
e
e
i
n
g
 
n
o
 
b
l
a
c
k
-
c
o
r
n
e
r
 
a
r
t
i
f
a
c
t
s
.

C
A
C
H
E
_
S
I
Z
E
 
=
 
1
3
6


#
 
S
h
o
w
 
b
e
f
o
r
e
/
a
f
t
e
r
 
f
o
r
 
a
 
f
e
w
 
e
x
a
m
p
l
e
 
i
m
a
g
e
s

n
_
c
o
m
p
a
r
e
 
=
 
4

r
n
g
 
=
 
n
p
.
r
a
n
d
o
m
.
d
e
f
a
u
l
t
_
r
n
g
(
4
2
)

c
o
m
p
a
r
e
_
i
d
x
 
=
 
r
n
g
.
c
h
o
i
c
e
(
l
e
n
(
g
z
)
,
 
s
i
z
e
=
n
_
c
o
m
p
a
r
e
,
 
r
e
p
l
a
c
e
=
F
a
l
s
e
)

c
o
m
p
a
r
e
_
i
d
s
 
=
 
g
a
l
a
x
y
_
i
d
s
[
c
o
m
p
a
r
e
_
i
d
x
]


i
m
a
g
e
s
_
b
e
f
o
r
e
 
=
 
[
_
l
o
a
d
_
i
m
a
g
e
(
I
M
A
G
E
_
D
I
R
 
/
 
f
"
{
g
i
d
}
.
j
p
g
"
)
 
f
o
r
 
g
i
d
 
i
n
 
c
o
m
p
a
r
e
_
i
d
s
]


f
r
o
m
 
u
g
d
a
t
a
l
a
b
.
m
o
d
e
l
s
.
g
a
l
a
x
y
_
z
o
o
.
i
m
a
g
e
s
_
p
i
p
e
l
i
n
e
 
i
m
p
o
r
t
 
_
c
r
o
p
_
c
e
n
t
e
r
,
 
_
r
e
s
i
z
e

i
m
a
g
e
s
_
a
f
t
e
r
 
=
 
[
_
r
e
s
i
z
e
(
_
c
r
o
p
_
c
e
n
t
e
r
(
i
m
g
,
 
C
R
O
P
_
F
R
A
C
T
I
O
N
)
,
 
C
A
C
H
E
_
S
I
Z
E
)
 
f
o
r
 
i
m
g
 
i
n
 
i
m
a
g
e
s
_
b
e
f
o
r
e
]


a
x
e
s
 
=
 
p
l
o
t
t
e
r
s
.
p
l
o
t
_
i
m
a
g
e
_
c
o
m
p
a
r
i
s
o
n
(
i
m
a
g
e
s
_
b
e
f
o
r
e
,
 
i
m
a
g
e
s
_
a
f
t
e
r
,
 
c
o
m
p
a
r
e
_
i
d
s
)

p
l
t
.
s
h
o
w
(
)


#
 
R
e
p
o
r
t
 
s
i
z
e
 
r
e
d
u
c
t
i
o
n

h
_
o
r
i
g
 
=
 
i
m
a
g
e
s
_
b
e
f
o
r
e
[
0
]
.
s
h
a
p
e
[
0
]

r
e
d
u
c
t
i
o
n
 
=
 
(
h
_
o
r
i
g
 
*
*
 
2
)
 
/
 
(
C
A
C
H
E
_
S
I
Z
E
 
*
*
 
2
)

p
r
i
n
t
(
f
"
O
r
i
g
i
n
a
l
:
 
{
h
_
o
r
i
g
}
x
{
h
_
o
r
i
g
}
 
=
 
{
h
_
o
r
i
g
*
*
2
:
,
}
 
p
i
x
e
l
s
"
)

p
r
i
n
t
(
f
"
A
f
t
e
r
 
c
r
o
p
+
r
e
s
i
z
e
:
 
{
C
A
C
H
E
_
S
I
Z
E
}
x
{
C
A
C
H
E
_
S
I
Z
E
}
 
=
 
{
C
A
C
H
E
_
S
I
Z
E
*
*
2
:
,
}
 
p
i
x
e
l
s
"
)

p
r
i
n
t
(
f
"
R
e
d
u
c
t
i
o
n
 
f
a
c
t
o
r
:
 
{
r
e
d
u
c
t
i
o
n
:
.
1
f
}
x
"
)

### Preprocess and save all images

We now crop and resize all images and save the result as a compressed numpy archive. This takes a few minutes but only needs to be done once — subsequent notebooks load the preprocessed arrays directly.

In [ ]:
g
z
_
i
m
a
g
e
s
 
=
 
G
a
l
a
x
y
Z
o
o
I
m
a
g
e
s
(

 
 
 
 
s
o
u
r
c
e
=
g
z
,

 
 
 
 
i
m
a
g
e
_
d
i
r
=
I
M
A
G
E
_
D
I
R
,

 
 
 
 
c
r
o
p
_
f
r
a
c
t
i
o
n
=
C
R
O
P
_
F
R
A
C
T
I
O
N
,

 
 
 
 
t
a
r
g
e
t
_
s
i
z
e
=
C
A
C
H
E
_
S
I
Z
E
,

)


p
r
i
n
t
(
f
"
P
r
e
p
r
o
c
e
s
s
e
d
 
i
m
a
g
e
s
 
s
h
a
p
e
:
 
{
g
z
_
i
m
a
g
e
s
.
i
m
a
g
e
s
.
s
h
a
p
e
}
"
)

p
r
i
n
t
(
f
"
M
e
m
o
r
y
:
 
{
g
z
_
i
m
a
g
e
s
.
i
m
a
g
e
s
.
n
b
y
t
e
s
 
/
 
1
e
9
:
.
2
f
}
 
G
B
"
)


n
p
.
s
a
v
e
z
_
c
o
m
p
r
e
s
s
e
d
(

 
 
 
 
"
a
r
t
i
f
a
c
t
s
/
g
a
l
a
x
y
_
z
o
o
_
i
m
a
g
e
s
.
n
p
z
"
,

 
 
 
 
i
m
a
g
e
s
=
g
z
_
i
m
a
g
e
s
.
i
m
a
g
e
s
,

 
 
 
 
g
a
l
a
x
y
_
i
d
s
=
g
a
l
a
x
y
_
i
d
s
,

)

p
r
i
n
t
(
"
S
a
v
e
d
 
a
r
t
i
f
a
c
t
s
/
g
a
l
a
x
y
_
z
o
o
_
i
m
a
g
e
s
.
n
p
z
"
)

## Task 12 — Train/Validation Split

We split the data 80/20 into training and validation sets using a random permutation with a fixed seed. After splitting, we compare the normalized label distributions of the two sets to ensure there are no systematic differences — the split should produce statistically indistinguishable distributions for all 37 labels.

In [ ]:
rng = np.random.default_rng(42)
idx = rng.permutation(len(gz))
n_train = int(0.8 * len(gz))
train_idx = np.sort(idx[:n_train])
val_idx = np.sort(idx[n_train:])
train_labels = gz.labels[train_idx]
val_labels = gz.labels[val_idx]

print(f"Training set: {len(train_idx)} galaxies ({len(train_idx)/len(gz)*100:.0f}%)")
print(f"Validation set: {len(val_idx)} galaxies ({len(val_idx)/len(gz)*100:.0f}%)")

# Compare distributions
axes = plotters.plot_split_distributions(
    train_labels, val_labels, LABEL_COLUMNS, LABEL_DESCRIPTIVE,
)
plt.show()

## Task 13 — Baseline Model

Before training any neural network, we establish a baseline: predict the training-set mean label for every image. This is the simplest possible model — it ignores the image entirely and always predicts the same label vector. Any useful CNN must outperform this baseline.

The loss function throughout this lab is the **root mean squared error** (RMSE), defined as:

$$L_{\mathrm{RMSE}} = \sqrt{\frac{1}{N_{\mathrm{galaxies}} \cdot N_{\mathrm{labels}}} \sum_i \sum_j (\ell_{\mathrm{true},ij} - \ell_{\mathrm{pred},ij})^2}$$

In [ ]:
train_rmse, val_rmse = baseline_rmse(train_labels, val_labels)
print(f"Baseline RMSE (mean prediction):")
print(f"  Training:   {train_rmse:.4f}")
print(f"  Validation: {val_rmse:.4f}")

### Save split indices and preprocessed data

In [ ]:
np.savez_compressed(
    "artifacts/split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    baseline_train_rmse=train_rmse,
    baseline_val_rmse=val_rmse,
)
print("Saved artifacts/split_indices.npz")
print(f"  train_idx: {train_idx.shape}")
print(f"  val_idx: {val_idx.shape}")